# Fall Detection - single-notebook run

Pose-based fall detection: Le2i videos -> MediaPipe keypoints -> normalized
features -> sliding windows -> BiLSTM + attention classifier.

This notebook is fully self-contained - every module is defined inline in a
cell below, so just open this single `.ipynb` in your Jupyter environment
and run top to bottom.

**Before you start:** make sure your Jupyter kernel/VM has GPU access
enabled - check with the GPU-check cell further down before starting the
long training run.

**You will need:** a Kaggle account + API token (Kaggle -> Settings -> API
-> Create New Token) to download the Le2i dataset - see the credentials
cell in section 2.

## 0. Install dependencies

We pin `mediapipe==0.10.21` and use its **legacy** `solutions.pose` API,
not the newer Tasks API (`PoseLandmarker`). We tried the Tasks API first
and hit native crashes on two different platforms for two different
reasons - macOS: a Metal GPU calculator crash; this Linux VM: the pose
graph touches GLES/EGL internals even in CPU-only mode, which segfaults
the whole kernel (no Python traceback) on a headless machine with no real
GPU/display. The legacy API needs none of that - no GL/EGL libraries, no
native crashes, verified across many real videos during development.

`mlflow<3` avoids `mlflow`'s own `protobuf>=6.31` requirement, which
conflicts with `mediapipe==0.10.21`'s `protobuf<5` requirement -
`mlflow<3` accepts the older protobuf mediapipe needs.

`numpy<2` is required by `mediapipe==0.10.21`'s wheel.

`--ignore-installed` guards against a common Debian/Ubuntu-base-image
quirk: a few packages (`blinker`, `cryptography`, ...) sometimes ship
pre-installed via `apt` with no pip `RECORD` file, so pip refuses to
*uninstall* them if something here wants a newer version - this just
tells pip to install over them instead.

In [ ]:
!pip install -q --ignore-installed kaggle "mediapipe==0.10.21" "numpy<2" "mlflow<3" scikit-learn tqdm opencv-python

In [ ]:
import copy
import json
import random
import urllib.parse
import zipfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import mlflow
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 1. Config

All hyperparameters and paths in one place - nothing hardcoded further down.

In [ ]:
CONFIG = {
    "paths": {
        "raw_dir": "data/raw",
        "processed_dir": "data/processed",
        "inventory_file": "data/inventory.json",
        "splits_file": "data/splits.json",
        "checkpoints_dir": "checkpoints",
        # mlflow>=3.x refuses the plain-directory filestore by default
        # (maintenance mode); sqlite is the supported local backend.
        "mlflow_uri": "sqlite:///mlruns.db",
    },
    "pose": {
        "model_complexity": 1,
        "min_detection_confidence": 0.5,
        "min_tracking_confidence": 0.5,
    },
    "windowing": {
        "window_size": 30,
        "stride": 15,
    },
    "model": {
        "hidden_dim": 128,
        "num_layers": 2,
        "bidirectional": True,
        "attention_dim": 64,
        "dropout": 0.3,
        "num_classes": 2,
    },
    "train": {
        "batch_size": 32,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_epochs": 100,
        "early_stopping_patience": 10,
        "seed": 42,
    },
}

## 2. Download the Le2i dataset

Source: Kaggle dataset `tuyenldvn/falldataset-imvia`, a mirror of the
original Le2i UMR6306 release - 190 videos (~17GB) across six scenes:
Coffee_room_01/02, Home_01/02 (per-frame fall annotations), Lecture_room
and Office (unannotated, treated as pure ADL/no-fall).

In [ ]:
import os
from pathlib import Path

# Fill these in with your own Kaggle credentials (kaggle.com -> Settings ->
# API -> Create New Token gives you a kaggle.json with these two fields).
# Don't commit this notebook anywhere with real values filled in here.
KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME"
KAGGLE_KEY = "YOUR_KAGGLE_KEY"

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(f'{{"username":"{KAGGLE_USERNAME}","key":"{KAGGLE_KEY}"}}')
os.chmod(kaggle_json, 0o600)
print(f"wrote {kaggle_json}")

In [ ]:
DATASET = "tuyenldvn/falldataset-imvia"


def _download_one_file(api, remote_name: str, dest_dir: Path) -> None:
    """Download a single dataset file, tolerating two Kaggle API quirks
    seen in practice: it sometimes wraps the file in a same-named .zip,
    and it sometimes saves the local filename percent-encoded (e.g.
    "video%20(1).txt" instead of "video (1).txt").
    """
    before = set(dest_dir.iterdir()) if dest_dir.exists() else set()
    dest_dir.mkdir(parents=True, exist_ok=True)
    api.dataset_download_file(DATASET, remote_name, path=str(dest_dir))
    new_files = [p for p in dest_dir.iterdir() if p not in before]

    for p in new_files:
        if p.suffix == ".zip":
            with zipfile.ZipFile(p) as z:
                z.extractall(dest_dir)
            p.unlink()

    for p in list(dest_dir.iterdir()):
        if "%" in p.name:
            p.rename(p.with_name(urllib.parse.unquote(p.name)))


def download_le2i_full(out_dir: Path) -> None:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    out_dir.mkdir(parents=True, exist_ok=True)
    api.dataset_download_files(DATASET, path=str(out_dir), unzip=True, quiet=False)


def download_le2i_annotations_only(out_dir: Path) -> None:
    """Just the per-frame ground-truth .txt files (~1MB total). Useful to
    sanity-check the pipeline before committing to the full ~17GB pull.
    """
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    all_files = []
    token = None
    while True:
        resp = api.dataset_list_files(DATASET, page_token=token)
        all_files.extend(resp.files)
        token = getattr(resp, "next_page_token", None) or getattr(resp, "nextPageToken", None)
        if not token:
            break

    annotation_files = [f for f in all_files if f.name.endswith(".txt") and "Annotation" in f.name]
    print(f"downloading {len(annotation_files)} annotation files to {out_dir} ...")
    for f in annotation_files:
        dest = out_dir / f.name
        if dest.exists():
            continue
        _download_one_file(api, f.name, dest.parent)

In [ ]:
# Full ~17GB download (190 videos). To sanity-check the pipeline on just a
# few videos first, comment this out and use download_le2i_annotations_only
# plus manually grabbing a couple of videos instead.
download_le2i_full(Path(CONFIG["paths"]["raw_dir"]))

## 3. M1 - dataset inventory and split decision

Walks the real Le2i layout (`<Scene>/<Scene>/Videos/video (N).avi` +
`Annotation_files/video (N).txt`, with `Coffee_room_02` typo'd as
`Annotations_files`, and `Lecture_room`/`Office` shipping no annotations
at all - treated as pure ADL scenes). Splits are stratified per
(scene, label) group at the video level rather than a full scene holdout:
with only six scenes (four fall-only, two ADL-only), holding a whole scene
out would either drop an entire environment from training or break class
balance.

In [ ]:
VIDEO_EXTS = {".avi", ".mp4", ".mkv", ".mov"}
ANNOTATION_DIR_NAMES = ("Annotation_files", "Annotations_files")

FALL = "fall"
ADL = "adl"


@dataclass
class VideoRecord:
    video_id: str
    scene: str
    path: str
    label: str
    has_annotation: bool
    annotation_path: str = None
    fall_start_frame: int = None
    fall_end_frame: int = None
    frame_count: int = None
    fps: float = None
    width: int = None
    height: int = None
    duration_sec: float = None
    warnings: list = field(default_factory=list)
    split: str = None


def _find_annotation_dir(scene_root: Path):
    for name in ANNOTATION_DIR_NAMES:
        candidate = scene_root / name
        if candidate.is_dir():
            return candidate
    return None


def _find_video_dir(scene_root: Path) -> Path:
    videos_dir = scene_root / "Videos"
    return videos_dir if videos_dir.is_dir() else scene_root


def parse_annotation(path: Path):
    lines = [ln.strip() for ln in path.read_text().splitlines() if ln.strip()]
    try:
        fall_start = int(lines[0])
        fall_end = int(lines[1])
        data_lines = lines[2:]
    except ValueError:
        # A handful of annotation files in the full 190-video Le2i release
        # don't carry the two-line fall_start/fall_end header (line 0 is
        # already a per-frame data row like "1,1,72,58,132,170"). Fall back
        # to treating the video as unannotated (ADL) instead of crashing
        # the whole inventory pass over one malformed file.
        fall_start = fall_end = 0
        data_lines = lines
    annotated_frames = len(data_lines)
    return fall_start, fall_end, annotated_frames


def _probe_video(path: Path):
    warnings = []
    try:
        import cv2
    except Exception as exc:
        return None, None, None, None, [f"opencv unavailable, skipped video probing: {exc}"]

    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        cap.release()
        return None, None, None, None, ["could not open video file"]

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
    fps = cap.get(cv2.CAP_PROP_FPS) or None
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or None
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or None
    cap.release()

    if not frame_count:
        warnings.append("frame count unavailable or zero")
    return frame_count, fps, width, height, warnings


def _scene_dirs(raw_dir: Path):
    scenes = []
    for top in sorted(p for p in raw_dir.iterdir() if p.is_dir()):
        children = [c for c in top.iterdir() if c.is_dir()]
        scene_root = children[0] if len(children) == 1 else top
        scenes.append((top.name, scene_root))
    return scenes


def discover_videos(raw_dir: Path, probe: bool = True):
    raw_dir = Path(raw_dir)
    records = []

    for scene_name, scene_root in _scene_dirs(raw_dir):
        annotation_dir = _find_annotation_dir(scene_root)
        video_dir = _find_video_dir(scene_root)

        video_paths = sorted(p for p in video_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)
        for video_path in video_paths:
            warnings = []
            annotation_path = None
            fall_start = fall_end = None
            label = ADL

            if annotation_dir is not None:
                candidate = annotation_dir / f"{video_path.stem}.txt"
                if candidate.exists():
                    annotation_path = candidate
                    fall_start, fall_end, annotated_frames = parse_annotation(candidate)
                    label = FALL if (fall_start or fall_end) else ADL
                    if annotated_frames <= 0:
                        warnings.append("annotation file has no per-frame rows")
                else:
                    warnings.append("scene has annotations but this video is missing one")

            frame_count = fps = width = height = None
            if probe:
                frame_count, fps, width, height, probe_warnings = _probe_video(video_path)
                warnings.extend(probe_warnings)
                if fall_end and frame_count and fall_end > frame_count:
                    warnings.append(
                        f"fall_end_frame ({fall_end}) exceeds probed frame_count ({frame_count})"
                    )

            duration_sec = frame_count / fps if frame_count and fps else None

            records.append(
                VideoRecord(
                    video_id=f"{scene_name}/{video_path.stem}",
                    scene=scene_name,
                    path=str(video_path.relative_to(raw_dir)),
                    label=label,
                    has_annotation=annotation_path is not None,
                    annotation_path=(
                        str(annotation_path.relative_to(raw_dir)) if annotation_path else None
                    ),
                    fall_start_frame=fall_start,
                    fall_end_frame=fall_end,
                    frame_count=frame_count,
                    fps=fps,
                    width=width,
                    height=height,
                    duration_sec=duration_sec,
                    warnings=warnings,
                )
            )

    return records


def assign_splits(records, train_frac: float = 0.7, val_frac: float = 0.15, seed: int = 42) -> None:
    rng = random.Random(seed)
    groups = {}
    for rec in records:
        groups.setdefault((rec.scene, rec.label), []).append(rec)

    for group_records in groups.values():
        shuffled = group_records[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = round(n * train_frac)
        n_val = round(n * val_frac)
        n_train = max(n_train, 1) if n else 0
        n_val = min(n_val, max(n - n_train, 0))

        for i, rec in enumerate(shuffled):
            if i < n_train:
                rec.split = "train"
            elif i < n_train + n_val:
                rec.split = "val"
            else:
                rec.split = "test"


def summarize(records):
    summary = {"total_videos": len(records), "by_scene": {}, "by_split": {}}
    for rec in records:
        scene_stats = summary["by_scene"].setdefault(rec.scene, {"fall": 0, "adl": 0, "warnings": 0})
        scene_stats[rec.label] += 1
        if rec.warnings:
            scene_stats["warnings"] += 1

        split_stats = summary["by_split"].setdefault(rec.split or "unassigned", {"fall": 0, "adl": 0})
        split_stats[rec.label] += 1
    return summary


def build_inventory(raw_dir, train_frac: float = 0.7, val_frac: float = 0.15, seed: int = 42, probe: bool = True):
    records = discover_videos(Path(raw_dir), probe=probe)
    if not records:
        raise FileNotFoundError(f"no video files found under {raw_dir}")
    assign_splits(records, train_frac=train_frac, val_frac=val_frac, seed=seed)
    return records, summarize(records)


def write_inventory(records, out_path) -> None:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps([asdict(r) for r in records], indent=2))


def write_splits(records, out_path) -> None:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    splits = {"train": [], "val": [], "test": []}
    for rec in records:
        splits.setdefault(rec.split or "unassigned", []).append(rec.video_id)
    out_path.write_text(json.dumps(splits, indent=2))

In [ ]:
records, summary = build_inventory(
    CONFIG["paths"]["raw_dir"], seed=CONFIG["train"]["seed"]
)
write_inventory(records, CONFIG["paths"]["inventory_file"])
write_splits(records, CONFIG["paths"]["splits_file"])

print(f"discovered {summary['total_videos']} videos\n")
print("by scene:")
for scene, stats in sorted(summary["by_scene"].items()):
    print(f"  {scene:16s} fall={stats['fall']:3d}  adl={stats['adl']:3d}  warnings={stats['warnings']}")
print("\nby split:")
for split, stats in summary["by_split"].items():
    print(f"  {split:12s} fall={stats['fall']:3d}  adl={stats['adl']:3d}")

## 4. M2 - pose extraction pipeline

MediaPipe `solutions.pose` (legacy API) -> hip-center/torso-scale
normalization -> per-frame feature vector (position, visibility, velocity,
torso angle) -> sliding windows labeled by overlap with the annotated fall
interval.

> If the very first pose-estimation call prints a one-line
> `AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'`,
> that's a harmless protobuf-version overlap with `mlflow` - confirmed
> during development across many real videos, it doesn't affect results
> and MediaPipe keeps working right after printing it.

In [ ]:
# --- pose.py (legacy mediapipe.solutions.pose API) ---

NUM_LANDMARKS = 33
NUM_CHANNELS = 4  # (x, y, z, visibility)

# Indices into the 33 MediaPipe pose landmarks used below.
NOSE = 0
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_HIP = 23
RIGHT_HIP = 24


def estimate_video(
    video_path: str,
    model_complexity: int = 1,
    min_detection_confidence: float = 0.5,
    min_tracking_confidence: float = 0.5,
) -> np.ndarray:
    """Run pose estimation over every frame of a video.

    Returns (T, 33, 4) of (x, y, z, visibility); frames with no detection
    are filled with NaN so downstream code can tell "missing" apart from a
    genuine (0, 0) landmark.
    """
    import cv2
    from mediapipe.python.solutions import pose as mp_pose

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"could not open video: {video_path}")

    frames = []
    try:
        with mp_pose.Pose(
            static_image_mode=False,
            model_complexity=model_complexity,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        ) as pose:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                result = pose.process(rgb)
                if result.pose_landmarks is None:
                    frames.append(np.full((NUM_LANDMARKS, NUM_CHANNELS), np.nan))
                else:
                    frames.append(
                        np.array(
                            [
                                [lm.x, lm.y, lm.z, lm.visibility]
                                for lm in result.pose_landmarks.landmark
                            ]
                        )
                    )
    finally:
        cap.release()

    if not frames:
        return np.empty((0, NUM_LANDMARKS, NUM_CHANNELS))
    return np.stack(frames, axis=0)

In [ ]:
# --- normalize.py ---

MIN_TORSO_SCALE = 1e-3


def interpolate_missing_frames(keypoints: np.ndarray) -> np.ndarray:
    """Linearly interpolate frames where pose estimation found no person.

    A dropped detection is far more likely than a person truly vanishing
    mid-clip (occlusion, motion blur during a fall - exactly the moment we
    care most about). Leading/trailing gaps use the nearest valid frame.
    """
    keypoints = keypoints.copy()
    t = keypoints.shape[0]
    valid = ~np.isnan(keypoints).any(axis=(1, 2))

    if not valid.any():
        return keypoints

    valid_idx = np.flatnonzero(valid)
    for i in range(t):
        if valid[i]:
            continue
        left = valid_idx[valid_idx < i]
        right = valid_idx[valid_idx > i]
        if left.size and right.size:
            lo, hi = left[-1], right[0]
            frac = (i - lo) / (hi - lo)
            keypoints[i] = keypoints[lo] * (1 - frac) + keypoints[hi] * frac
        elif right.size:
            keypoints[i] = keypoints[right[0]]
        else:
            keypoints[i] = keypoints[left[-1]]

    return keypoints


def normalize_pose_sequence(keypoints: np.ndarray) -> np.ndarray:
    """Translate by hip-center and scale by torso length, per frame - makes
    the representation invariant to a person's position/distance from
    camera, which varies across scenes and carries no fall information.
    """
    hip_center = (keypoints[:, LEFT_HIP, :3] + keypoints[:, RIGHT_HIP, :3]) / 2
    shoulder_center = (keypoints[:, LEFT_SHOULDER, :3] + keypoints[:, RIGHT_SHOULDER, :3]) / 2
    torso_scale = np.linalg.norm(shoulder_center - hip_center, axis=-1)
    torso_scale = np.maximum(torso_scale, MIN_TORSO_SCALE)

    normalized = keypoints.copy()
    normalized[:, :, :3] = (keypoints[:, :, :3] - hip_center[:, None, :]) / torso_scale[:, None, None]
    return normalized

In [ ]:
# --- features.py ---

# Image y increases downward, so an upright torso vector (hip -> shoulder)
# points in -y; that is the reference torso tilt is measured against.
UP_VECTOR = np.array([0.0, -1.0])


def compute_velocity(sequence: np.ndarray) -> np.ndarray:
    """Frame-to-frame finite difference; first frame is zero rather than
    dropped, so the output keeps the same length as the input."""
    velocity = np.zeros_like(sequence)
    velocity[1:] = sequence[1:] - sequence[:-1]
    return velocity


def compute_torso_angle(keypoints: np.ndarray) -> np.ndarray:
    """Angle (radians, [0, pi]) between torso and vertical: 0 = upright,
    pi/2 = lying flat - the single strongest per-frame fall signal here.
    """
    hip_center = (keypoints[:, LEFT_HIP, :2] + keypoints[:, RIGHT_HIP, :2]) / 2
    shoulder_center = (keypoints[:, LEFT_SHOULDER, :2] + keypoints[:, RIGHT_SHOULDER, :2]) / 2
    torso = shoulder_center - hip_center

    norm = np.linalg.norm(torso, axis=-1)
    norm = np.maximum(norm, 1e-8)
    cos_angle = (torso @ UP_VECTOR) / norm
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.arccos(cos_angle)


def assemble_feature_vector(keypoints: np.ndarray) -> np.ndarray:
    """Per-frame feature matrix fed to the model: (T, D), D = 33*2 (xy)
    + 33 (visibility) + 33*2 (xy velocity) + 1 (torso angle)
    + 1 (torso angular velocity) = 167.
    """
    xy = keypoints[:, :, :2].reshape(keypoints.shape[0], NUM_LANDMARKS * 2)
    visibility = keypoints[:, :, 3]
    xy_velocity = compute_velocity(xy)

    torso_angle = compute_torso_angle(keypoints)
    torso_angular_velocity = compute_velocity(torso_angle[:, None])[:, 0]

    return np.concatenate(
        [xy, visibility, xy_velocity, torso_angle[:, None], torso_angular_velocity[:, None]],
        axis=1,
    )


FEATURE_DIM = NUM_LANDMARKS * 2 + NUM_LANDMARKS + NUM_LANDMARKS * 2 + 1 + 1

In [ ]:
# --- windowing.py ---

def sliding_windows(features: np.ndarray, window_size: int, stride: int):
    """Chop a (T, D) feature sequence into overlapping fixed-size windows.
    A video shorter than window_size yields zero windows rather than a
    padded partial one.
    """
    t = features.shape[0]
    if t < window_size:
        return np.empty((0, window_size, features.shape[1])), np.empty((0, 2), dtype=int)

    starts = np.arange(0, t - window_size + 1, stride)
    windows = np.stack([features[s : s + window_size] for s in starts])
    frame_ranges = np.stack([starts, starts + window_size - 1], axis=1)
    return windows, frame_ranges


def label_windows(frame_ranges: np.ndarray, fall_start_frame, fall_end_frame) -> np.ndarray:
    """Label a window 1 if it overlaps the annotated fall interval at all,
    else 0. Le2i frame numbers are 1-indexed; frame_ranges are 0-indexed.
    """
    if not fall_start_frame and not fall_end_frame:
        return np.zeros(len(frame_ranges), dtype=int)

    fall_start = fall_start_frame - 1
    fall_end = fall_end_frame - 1
    starts, ends = frame_ranges[:, 0], frame_ranges[:, 1]
    overlaps = ~((ends < fall_start) | (starts > fall_end))
    return overlaps.astype(int)

In [ ]:
# --- pipeline.py ---

def process_video(
    video_path: str,
    fall_start_frame=None,
    fall_end_frame=None,
    *,
    model_complexity: int = 1,
    min_detection_confidence: float = 0.5,
    min_tracking_confidence: float = 0.5,
    window_size: int = 30,
    stride: int = 15,
) -> dict:
    """Full M2 pipeline on one video: pose -> normalize -> features -> windows."""
    raw_keypoints = estimate_video(
        video_path,
        model_complexity=model_complexity,
        min_detection_confidence=min_detection_confidence,
        min_tracking_confidence=min_tracking_confidence,
    )
    num_frames = raw_keypoints.shape[0]
    if num_frames == 0:
        return {
            "windows": np.empty((0, window_size, FEATURE_DIM)),
            "labels": np.empty((0,), dtype=int),
            "frame_ranges": np.empty((0, 2), dtype=int),
            "num_frames": 0,
            "detection_rate": 0.0,
        }

    missing = np.isnan(raw_keypoints).any(axis=(1, 2))
    detection_rate = 1.0 - missing.mean()

    keypoints = interpolate_missing_frames(raw_keypoints)
    keypoints = normalize_pose_sequence(keypoints)
    feature_matrix = assemble_feature_vector(keypoints)

    windows, frame_ranges = sliding_windows(feature_matrix, window_size, stride)
    labels = label_windows(frame_ranges, fall_start_frame, fall_end_frame)

    return {
        "windows": windows,
        "labels": labels,
        "frame_ranges": frame_ranges,
        "num_frames": num_frames,
        "detection_rate": detection_rate,
    }

### Run extraction over every video in the inventory

In [ ]:
inventory = json.loads(Path(CONFIG["paths"]["inventory_file"]).read_text())
processed_dir = Path(CONFIG["paths"]["processed_dir"])
processed_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path(CONFIG["paths"]["raw_dir"])

pose_cfg = CONFIG["pose"]
window_cfg = CONFIG["windowing"]

total_windows = 0
total_positive = 0
low_detection = []

for i, record in enumerate(inventory, start=1):
    video_path = raw_dir / record["path"]
    out_path = processed_dir / f"{record['video_id'].replace('/', '__')}.npz"

    print(f"[{i}/{len(inventory)}] {record['video_id']} ({record['label']}) ...", end=" ")

    result = process_video(
        str(video_path),
        fall_start_frame=record.get("fall_start_frame"),
        fall_end_frame=record.get("fall_end_frame"),
        model_complexity=pose_cfg["model_complexity"],
        min_detection_confidence=pose_cfg["min_detection_confidence"],
        min_tracking_confidence=pose_cfg["min_tracking_confidence"],
        window_size=window_cfg["window_size"],
        stride=window_cfg["stride"],
    )

    np.savez_compressed(
        out_path,
        windows=result["windows"],
        labels=result["labels"],
        frame_ranges=result["frame_ranges"],
        scene=record["scene"],
        split=record["split"],
        video_id=record["video_id"],
    )

    n_windows = result["windows"].shape[0]
    n_positive = int(result["labels"].sum())
    total_windows += n_windows
    total_positive += n_positive
    if result["detection_rate"] < 0.8:
        low_detection.append((record["video_id"], result["detection_rate"]))

    print(
        f"{result['num_frames']} frames, detection_rate={result['detection_rate']:.2f}, "
        f"{n_windows} windows ({n_positive} positive) -> {out_path.name}"
    )

print(f"\nwrote {len(inventory)} .npz files to {processed_dir}")
print(f"total windows: {total_windows} ({total_positive} positive, {total_windows - total_positive} negative)")
if low_detection:
    print(f"\n{len(low_detection)} video(s) had pose detection_rate < 0.8:")
    for video_id, rate in low_detection:
        print(f"  {video_id}: {rate:.2f}")

## 5. M3 - dataset, model, training, evaluation

BiLSTM over the 167-dim per-frame features with a learned attention-pooling
head, so the classifier weighs the frames that actually look like a fall
instead of averaging them out over a mostly-idle window.

In [ ]:
# --- dataset.py ---

class FallWindowDataset(Dataset):
    """Loads windows/labels from the per-video .npz files, filtered to one
    split ("train"/"val"/"test") - the split each .npz carries was decided
    once, in M1's inventory step.
    """

    def __init__(self, processed_dir, split: str):
        self.processed_dir = Path(processed_dir)
        self.split = split

        windows, labels, video_ids = [], [], []
        for npz_path in sorted(self.processed_dir.glob("*.npz")):
            data = np.load(npz_path, allow_pickle=True)
            if str(data["split"]) != split or data["windows"].shape[0] == 0:
                continue
            windows.append(data["windows"])
            labels.append(data["labels"])
            video_ids.extend([str(data["video_id"])] * data["windows"].shape[0])

        if windows:
            self.windows = np.concatenate(windows, axis=0).astype(np.float32)
            self.labels = np.concatenate(labels, axis=0).astype(np.int64)
        else:
            self.windows = np.empty((0, 0, 0), dtype=np.float32)
            self.labels = np.empty((0,), dtype=np.int64)
        self.video_ids = video_ids

    def __len__(self) -> int:
        return self.windows.shape[0]

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.windows[idx]), torch.tensor(self.labels[idx])

    @property
    def input_dim(self) -> int:
        return self.windows.shape[-1] if self.windows.size else 0

    def class_counts(self) -> dict:
        values, counts = np.unique(self.labels, return_counts=True)
        return dict(zip(values.tolist(), counts.tolist()))


def make_class_weights(dataset: FallWindowDataset, num_classes: int = 2) -> torch.Tensor:
    """Inverse-frequency class weights: falls are a small fraction of
    frames even inside a fall-labeled video, so an unweighted loss would
    let the model coast to high accuracy while never predicting "fall".
    """
    counts = dataset.class_counts()
    weights = torch.ones(num_classes)
    total = sum(counts.values())
    if total == 0:
        return weights
    for cls in range(num_classes):
        count = counts.get(cls, 0)
        weights[cls] = total / (num_classes * count) if count > 0 else 0.0
    return weights

In [ ]:
# --- model.py ---

class AttentionPooling(nn.Module):
    """Learns a scalar importance score per timestep and returns the
    weighted sum of the sequence."""

    def __init__(self, input_dim: int, attention_dim: int):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(input_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1),
        )

    def forward(self, sequence: torch.Tensor):
        scores = self.score(sequence).squeeze(-1)  # (batch, time)
        weights = torch.softmax(scores, dim=-1)
        pooled = torch.bmm(weights.unsqueeze(1), sequence).squeeze(1)
        return pooled, weights


class FallDetectionModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 2,
        bidirectional: bool = True,
        attention_dim: int = 64,
        dropout: float = 0.3,
        num_classes: int = 2,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.attention = AttentionPooling(lstm_out_dim, attention_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_out_dim, num_classes)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        sequence_out, _ = self.lstm(x)
        pooled, weights = self.attention(sequence_out)
        logits = self.classifier(self.dropout(pooled))
        if return_attention:
            return logits, weights
        return logits

    @classmethod
    def from_config(cls, config: dict, input_dim: int) -> "FallDetectionModel":
        model_cfg = config["model"]
        return cls(
            input_dim=input_dim,
            hidden_dim=model_cfg["hidden_dim"],
            num_layers=model_cfg["num_layers"],
            bidirectional=model_cfg["bidirectional"],
            attention_dim=model_cfg["attention_dim"],
            dropout=model_cfg["dropout"],
            num_classes=model_cfg["num_classes"],
        )

In [ ]:
# --- train.py ---

def _run_epoch(model, loader, criterion, optimizer, device):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.enable_grad() if is_train else torch.no_grad():
        for windows, labels in loader:
            windows, labels = windows.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()

            logits = model(windows)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * windows.size(0)
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += windows.size(0)

    if total == 0:
        return 0.0, 0.0
    return total_loss / total, correct / total


def train(config: dict) -> dict:
    """Train one config end to end; returns the run summary and writes
    the best-val-loss checkpoint to `paths.checkpoints_dir`.
    """
    paths = config["paths"]
    train_cfg = config["train"]

    torch.manual_seed(train_cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"training on {device}")

    train_ds = FallWindowDataset(paths["processed_dir"], "train")
    val_ds = FallWindowDataset(paths["processed_dir"], "val")
    if len(train_ds) == 0:
        raise ValueError(
            f"no training windows found under {paths['processed_dir']} - run the extraction cell first"
        )

    train_loader = DataLoader(train_ds, batch_size=train_cfg["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=train_cfg["batch_size"], shuffle=False)

    model = FallDetectionModel.from_config(config, input_dim=train_ds.input_dim).to(device)
    class_weights = make_class_weights(train_ds).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=train_cfg["lr"], weight_decay=train_cfg["weight_decay"]
    )

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    history = []

    mlflow.set_tracking_uri(paths["mlflow_uri"])
    with mlflow.start_run():
        mlflow.log_params(
            {
                "input_dim": train_ds.input_dim,
                "train_windows": len(train_ds),
                "val_windows": len(val_ds),
                **{f"model.{k}": v for k, v in config["model"].items()},
                **{f"train.{k}": v for k, v in train_cfg.items()},
            }
        )

        for epoch in range(1, train_cfg["max_epochs"] + 1):
            train_loss, train_acc = _run_epoch(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc = _run_epoch(model, val_loader, criterion, None, device)

            mlflow.log_metrics(
                {"train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc},
                step=epoch,
            )
            history.append(
                {"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc}
            )
            print(
                f"epoch {epoch:3d} | train_loss {train_loss:.4f} acc {train_acc:.3f} "
                f"| val_loss {val_loss:.4f} acc {val_acc:.3f}"
            )

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= train_cfg["early_stopping_patience"]:
                    print(f"early stopping at epoch {epoch}")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        checkpoint_dir = Path(paths["checkpoints_dir"])
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_path = checkpoint_dir / "best_model.pt"
        torch.save(
            {"model_state_dict": model.state_dict(), "input_dim": train_ds.input_dim, "config": config},
            checkpoint_path,
        )
        mlflow.log_artifact(str(checkpoint_path))

    return {"best_val_loss": best_val_loss, "history": history, "checkpoint_path": str(checkpoint_path)}

In [ ]:
# --- evaluate.py ---

def evaluate(config: dict, checkpoint_path) -> dict:
    paths = config["paths"]
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    test_ds = FallWindowDataset(paths["processed_dir"], "test")
    if len(test_ds) == 0:
        raise ValueError(
            f"no test windows found under {paths['processed_dir']} - check the split assignment"
        )

    model = FallDetectionModel.from_config(checkpoint["config"], input_dim=checkpoint["input_dim"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    loader = DataLoader(test_ds, batch_size=config["train"]["batch_size"], shuffle=False)

    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for windows, labels in loader:
            logits = model(windows)
            probs = torch.softmax(logits, dim=-1)[:, 1]
            preds = logits.argmax(dim=-1)
            all_labels.extend(labels.tolist())
            all_preds.extend(preds.tolist())
            all_probs.extend(probs.tolist())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    report = classification_report(
        all_labels, all_preds, target_names=["adl", "fall"], output_dict=True, zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    pr_auc = (
        average_precision_score(all_labels, all_probs)
        if len(set(all_labels.tolist())) > 1
        else float("nan")
    )

    return {
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "pr_auc": pr_auc,
        "num_test_windows": len(test_ds),
    }

### Confirm the GPU is visible, then train + evaluate

In [ ]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU visible to torch - check your Jupyter environment's GPU/driver setup")

In [ ]:
result = train(CONFIG)
print(f"\nbest val_loss: {result['best_val_loss']:.4f}")
print(f"checkpoint: {result['checkpoint_path']}")

try:
    metrics = evaluate(CONFIG, result["checkpoint_path"])
    print(f"\ntest set ({metrics['num_test_windows']} windows):")
    print(json.dumps(metrics["classification_report"], indent=2))
    print("confusion matrix [[TN, FP], [FN, TP]]:", metrics["confusion_matrix"])
    print(f"PR-AUC: {metrics['pr_auc']:.4f}")
except ValueError as e:
    print(f"\nskipping test evaluation: {e}")

## 6. Results

Everything above is written to the local filesystem: checkpoint at
`checkpoints/best_model.pt`, MLflow run data in `mlruns.db` + `mlruns/`,
and `data/inventory.json` / `data/splits.json`. If this Jupyter environment
is ephemeral (a container/VM that gets torn down), copy those paths
somewhere durable before the session ends - swap the cell below for
whatever copy mechanism your environment provides.

In [ ]:
import shutil
from pathlib import Path

# Example: copy results into a single archive folder. Replace this with an
# upload to your own storage (S3/GCS/scp/rsync/etc.) if this environment
# doesn't persist local files across sessions.
results_dir = Path("fall-detection-results")
results_dir.mkdir(exist_ok=True)
for p in ["checkpoints", "mlruns.db", "mlruns", "data/inventory.json", "data/splits.json"]:
    src = Path(p)
    if not src.exists():
        continue
    dest = results_dir / src.name
    if src.is_dir():
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dest)
print(f"copied results into {results_dir.resolve()}")